# Plan a Use Model runtime safely

Read the public runtime registry, inspect local hardware, and plan one real reviewed model before acquisition. Compatibility is evidence for planning—not a benchmark, safety certification, or zero-out-of-memory guarantee.

In [ ]:
import json
import os
from urllib.request import Request, urlopen

ORIGIN = "https://superii.site"
request = Request(f"{ORIGIN}/runtime-registry.json", headers={"Accept": "application/json"})
with urlopen(request, timeout=20) as response:
    registry = json.load(response)
print(json.dumps(registry, indent=2)[:5000])

## Match a real revision to this machine

Install `superii-sdk==0.2.2`, then set `SUPERII_REPOSITORY` to an actual `owner/slug`. Inspection and planning do not execute model code. Unsupported or ambiguous formats fail with a reason.

In [ ]:
import superii

repository = os.getenv("SUPERII_REPOSITORY", "").strip()
machine = superii.hardware()
if repository:
    manifest = superii.inspect(repository)
    selected = superii.plan(manifest, machine=machine, context_size=4096)
    print(selected)
else:
    manifest = selected = None
    print("Choose a reviewed live model and set SUPERII_REPOSITORY.")

## Acquire and run only after review

Review the immutable revision, license, selected files, byte estimate, runtime, and trust boundary. The default below performs no download and starts no persistent inference service.

In [ ]:
ALLOW_ACQUISITION = False
ALLOW_LOCAL_INFERENCE = False

if ALLOW_ACQUISITION and repository and selected is not None:
    snapshot = superii.pull(repository, revision=selected.revision, files=tuple(selected.files))
    if not superii.verify(snapshot):
        raise RuntimeError("Verified acquisition failed closed")
    print(f"Verified snapshot: {snapshot.path}")
    if ALLOW_LOCAL_INFERENCE:
        with superii.load(repository, revision=selected.revision) as model:
            print(model.generate("Explain your intended use limits.", max_tokens=128))
else:
    print("No model bytes acquired and no inference started.")